# Experiment 2 — Sparse Architectural Constraints

Visualization-only notebook. All heavy compute runs on Modal; this notebook loads JSON/npz artifacts.

**Two approaches tested:**
- **L1 activation penalty** (soft): L1 on post-GELU d_ff hidden activations at λ=1e-4/1e-3/1e-2
- **Top-K MLP** (hard): zero all-but-top-k neurons per token at k=10%/25%/50% of d_ff=1536

## Download artifacts first

```bash
# Baseline (from Experiment 1)
modal volume get tti-checkpoints eval/expt1_baseline_seed42/ checkpoints/eval/expt1_baseline_seed42/

# L1 runs
modal volume get tti-checkpoints eval/expt2_l1_1e-4_seed42/ checkpoints/eval/expt2_l1_1e-4_seed42/
modal volume get tti-checkpoints eval/expt2_l1_1e-3_seed42/ checkpoints/eval/expt2_l1_1e-3_seed42/
modal volume get tti-checkpoints eval/expt2_l1_1e-2_seed42/ checkpoints/eval/expt2_l1_1e-2_seed42/

# Top-K runs
modal volume get tti-checkpoints eval/expt2_topk_10pct_seed42/ checkpoints/eval/expt2_topk_10pct_seed42/
modal volume get tti-checkpoints eval/expt2_topk_25pct_seed42/ checkpoints/eval/expt2_topk_25pct_seed42/
modal volume get tti-checkpoints eval/expt2_topk_50pct_seed42/ checkpoints/eval/expt2_topk_50pct_seed42/
```

Local deps:
```bash
pip install matplotlib seaborn pandas
```

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np
import pandas as pd
import seaborn as sns

ARTIFACT_DIR = "../checkpoints/eval"

RUN_NAMES = [
    "expt1_baseline_seed42",
    "expt2_l1_1e-4_seed42",
    "expt2_l1_1e-3_seed42",
    "expt2_l1_1e-2_seed42",
    "expt2_topk_10pct_seed42",
    "expt2_topk_25pct_seed42",
    "expt2_topk_50pct_seed42",
]
RUN_LABELS = {
    "expt1_baseline_seed42":    "Baseline",
    "expt2_l1_1e-4_seed42":     "L1 λ=1e-4",
    "expt2_l1_1e-3_seed42":     "L1 λ=1e-3",
    "expt2_l1_1e-2_seed42":     "L1 λ=1e-2",
    "expt2_topk_10pct_seed42":  "Top-K 10%",
    "expt2_topk_25pct_seed42":  "Top-K 25%",
    "expt2_topk_50pct_seed42":  "Top-K 50%",
}
# Grey baseline; green gradient for L1; purple gradient for Top-K
RUN_COLORS = {
    "expt1_baseline_seed42":    "#555555",
    "expt2_l1_1e-4_seed42":     "#74c476",
    "expt2_l1_1e-3_seed42":     "#31a354",
    "expt2_l1_1e-2_seed42":     "#006d2c",
    "expt2_topk_10pct_seed42":  "#9e9ac8",
    "expt2_topk_25pct_seed42":  "#756bb1",
    "expt2_topk_50pct_seed42":  "#54278f",
}
RUN_LINESTYLES = {
    "expt1_baseline_seed42":    "--",
    "expt2_l1_1e-4_seed42":     "-",
    "expt2_l1_1e-3_seed42":     "-",
    "expt2_l1_1e-2_seed42":     "-",
    "expt2_topk_10pct_seed42":  "-",
    "expt2_topk_25pct_seed42":  "-",
    "expt2_topk_50pct_seed42":  "-",
}
N_LAYERS = 6


def artifact_path(run_name, filename):
    return os.path.join(ARTIFACT_DIR, run_name, filename)

def load_json(run_name, filename):
    with open(artifact_path(run_name, filename)) as f:
        return json.load(f)

def load_npz(run_name, filename):
    return np.load(artifact_path(run_name, filename))

def has_artifact(run_name, filename):
    return os.path.exists(artifact_path(run_name, filename))


AVAILABLE_RUNS = [r for r in RUN_NAMES if has_artifact(r, "perplexity.json")]
print(f"Available runs ({len(AVAILABLE_RUNS)}/{len(RUN_NAMES)}):")
for r in AVAILABLE_RUNS:
    print(f"  {RUN_LABELS[r]}")

## 1. Perplexity

Key question: does enforcing sparsity hurt language modelling quality?

In [ ]:
rows = []
for run_name in AVAILABLE_RUNS:
    data = load_json(run_name, "perplexity.json")
    rows.append({
        "Run": RUN_LABELS[run_name],
        "Val Loss": f"{data['val_loss']:.4f}",
        "Perplexity": f"{data['perplexity']:.2f}",
    })

df = pd.DataFrame(rows).set_index("Run")
display(df)

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4))
vals = [float(r["Perplexity"]) for r in rows]
colors = [RUN_COLORS[r] for r in AVAILABLE_RUNS]
bars = ax.bar([RUN_LABELS[r] for r in AVAILABLE_RUNS], vals, color=colors)
ax.set_ylabel("Perplexity (lower is better)")
ax.set_title("Validation Perplexity — Experiment 2")
ax.tick_params(axis="x", rotation=30)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f"{v:.2f}",
            ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

## 2. Activation Sparsity

Fraction of d_ff hidden neurons with |activation| < 0.01, per layer.

- **Top-K runs**: sparsity is exact by construction (10%/25%/50% zeroed = 90%/75%/50% inactive)
- **L1 runs**: soft sparsity — fraction depends on how much the penalty pushes activations to zero
- **Baseline**: near-zero from GELU, but no pressure

In [ ]:
sparsity_runs = [r for r in AVAILABLE_RUNS if has_artifact(r, "sparsity.json")]

if not sparsity_runs:
    print("No sparsity.json artifacts found yet. Run eval_main first.")
else:
    sparsity_data = {r: load_json(r, "sparsity.json") for r in sparsity_runs}

    # --- Line chart: sparsity per layer ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for run_name in sparsity_runs:
        d = sparsity_data[run_name]
        layers = list(range(N_LAYERS))
        vals = [d.get(f"layer_{i}", 0) for i in layers]
        kw = dict(
            marker="o",
            label=RUN_LABELS[run_name],
            color=RUN_COLORS[run_name],
            linestyle=RUN_LINESTYLES[run_name],
        )
        axes[0].plot(layers, vals, **kw)

    axes[0].set_xlabel("Layer")
    axes[0].set_ylabel("Fraction inactive (|act| < 0.01)")
    axes[0].set_title("MLP Hidden Sparsity per Layer")
    axes[0].set_xticks(range(N_LAYERS))
    axes[0].set_xticklabels([f"L{i}" for i in range(N_LAYERS)])
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)

    # --- Bar chart: mean sparsity ---
    means = [sparsity_data[r]["mean"] for r in sparsity_runs]
    colors = [RUN_COLORS[r] for r in sparsity_runs]
    bars = axes[1].bar([RUN_LABELS[r] for r in sparsity_runs], means, color=colors)
    axes[1].set_ylabel("Mean fraction inactive")
    axes[1].set_title("Mean Sparsity Across Layers")
    axes[1].tick_params(axis="x", rotation=30)
    for bar, v in zip(bars, means):
        axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                     f"{v:.3f}", ha="center", va="bottom", fontsize=8)

    plt.tight_layout()
    plt.show()

    # Summary table
    rows = []
    for run_name in sparsity_runs:
        d = sparsity_data[run_name]
        row = {"Run": RUN_LABELS[run_name], "Mean": f"{d['mean']:.3f}"}
        for i in range(N_LAYERS):
            row[f"L{i}"] = f"{d.get(f'layer_{i}', 0):.3f}"
        rows.append(row)
    display(pd.DataFrame(rows).set_index("Run"))

## 3. Polysemanticity

Per-neuron entropy of the token-type activation distribution (measured on mlp_hidden).

- **High entropy** = neuron responds to many different token types → **polysemantic**
- **Low entropy** = neuron responds consistently to a narrow set of tokens → **monosemantic**

L1 counterintuitively *increases* polysemanticity by spreading activation budget across more tokens. Top-K enforces selectivity, driving neurons toward monosemanticity.

In [ ]:
poly_runs = [r for r in AVAILABLE_RUNS if has_artifact(r, "polysemanticity.json")]

if not poly_runs:
    print("No polysemanticity.json artifacts found. Run eval_main first.")
else:
    poly_data = {r: load_json(r, "polysemanticity.json") for r in poly_runs}

    # --- Line chart: entropy per layer ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for run_name in poly_runs:
        d = poly_data[run_name]
        layers = list(range(N_LAYERS))
        vals = [d.get(f"layer_{i}", 0) for i in layers]
        axes[0].plot(
            layers, vals,
            marker="o",
            label=RUN_LABELS[run_name],
            color=RUN_COLORS[run_name],
            linestyle=RUN_LINESTYLES[run_name],
        )

    axes[0].set_xlabel("Layer")
    axes[0].set_ylabel("Mean per-neuron entropy (nats)")
    axes[0].set_title("Polysemanticity per Layer (lower = more monosemantic)")
    axes[0].set_xticks(range(N_LAYERS))
    axes[0].set_xticklabels([f"L{i}" for i in range(N_LAYERS)])
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)

    # --- Bar chart: mean polysemanticity ---
    means = [poly_data[r]["mean"] for r in poly_runs]
    colors = [RUN_COLORS[r] for r in poly_runs]
    bars = axes[1].bar([RUN_LABELS[r] for r in poly_runs], means, color=colors)
    axes[1].set_ylabel("Mean entropy (nats)")
    axes[1].set_title("Mean Polysemanticity Across Layers")
    axes[1].tick_params(axis="x", rotation=30)
    for bar, v in zip(bars, means):
        axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                     f"{v:.3f}", ha="center", va="bottom", fontsize=8)

    plt.tight_layout()
    plt.show()

    # Summary table
    rows = []
    for run_name in poly_runs:
        d = poly_data[run_name]
        row = {"Run": RUN_LABELS[run_name], "Mean": f"{d['mean']:.3f}"}
        for i in range(N_LAYERS):
            row[f"L{i}"] = f"{d.get(f'layer_{i}', 0):.3f}"
        rows.append(row)
    display(pd.DataFrame(rows).set_index("Run"))

    # --- Delta vs baseline ---
    if "expt1_baseline_seed42" in poly_data:
        base = poly_data["expt1_baseline_seed42"]
        print("\nChange vs baseline (%):")
        for run_name in poly_runs:
            if run_name == "expt1_baseline_seed42":
                continue
            delta = (poly_data[run_name]["mean"] - base["mean"]) / (base["mean"] + 1e-10) * 100
            sign = "+" if delta > 0 else ""
            print(f"  {RUN_LABELS[run_name]:20s}  {sign}{delta:.1f}%")

## 4. Neuron Utilization

For each neuron, what fraction of tokens does it activate on (|act| > 0.01)?

- **Dead neurons** (< 1% of tokens): wasted capacity
- **Always-on neurons** (> 99% of tokens): no selectivity — effectively a bias term
- **Selective neurons** (1–99%): doing meaningful work

Top-K should eliminate always-on neurons by construction (forced off for low-magnitude tokens). L1 may create dead neurons if the penalty pushes activations to zero everywhere.

In [ ]:
util_runs = [r for r in AVAILABLE_RUNS if has_artifact(r, "neuron_utilization.json")]

if not util_runs:
    print("No neuron_utilization.json found. Run eval_main first.")
else:
    util_data = {r: load_json(r, "neuron_utilization.json") for r in util_runs}

    # --- Summary table: dead / always-on fractions ---
    rows = []
    for run_name in util_runs:
        d = util_data[run_name]
        row = {"Run": RUN_LABELS[run_name]}
        for i in range(N_LAYERS):
            key = str(i)
            if key in d:
                row[f"L{i} dead%"] = f"{d[key]['frac_dead']*100:.1f}"
                row[f"L{i} on%"] = f"{d[key]['frac_always_on']*100:.1f}"
        rows.append(row)
    display(pd.DataFrame(rows).set_index("Run"))

    # --- Bar charts: dead & always-on per layer ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for run_name in util_runs:
        d = util_data[run_name]
        layers = [i for i in range(N_LAYERS) if str(i) in d]
        dead_vals  = [d[str(i)]["frac_dead"] * 100       for i in layers]
        alwon_vals = [d[str(i)]["frac_always_on"] * 100  for i in layers]
        kw = dict(marker="o", label=RUN_LABELS[run_name],
                  color=RUN_COLORS[run_name], linestyle=RUN_LINESTYLES[run_name])
        axes[0].plot(layers, dead_vals,  **kw)
        axes[1].plot(layers, alwon_vals, **kw)

    for ax, title, ylabel in zip(
        axes,
        ["Dead Neurons per Layer (< 1% active)", "Always-On Neurons per Layer (> 99% active)"],
        ["% of d_ff neurons", "% of d_ff neurons"],
    ):
        ax.set_xlabel("Layer")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.set_xticks(range(N_LAYERS))
        ax.set_xticklabels([f"L{i}" for i in range(N_LAYERS)])
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    # --- Per-neuron utilization histograms (npz) ---
    npz_runs = [r for r in util_runs if has_artifact(r, "neuron_utilization.npz")]
    if npz_runs:
        fig, axes = plt.subplots(len(npz_runs), N_LAYERS,
                                 figsize=(N_LAYERS * 2.5, len(npz_runs) * 2.2),
                                 squeeze=False)
        for row_idx, run_name in enumerate(npz_runs):
            npz = load_npz(run_name, "neuron_utilization.npz")
            for col_idx in range(N_LAYERS):
                ax = axes[row_idx][col_idx]
                key = f"layer_{col_idx}"
                if key not in npz.files:
                    ax.axis("off")
                    continue
                frac = npz[key]  # shape [d_ff]
                ax.hist(frac, bins=40, color=RUN_COLORS[run_name], alpha=0.8, edgecolor="none")
                ax.axvline(0.01, color="red",   linewidth=1, linestyle="--", label="dead (<1%)")
                ax.axvline(0.99, color="blue",  linewidth=1, linestyle="--", label="always-on (>99%)")
                if col_idx == 0:
                    ax.set_ylabel(RUN_LABELS[run_name], fontsize=8)
                if row_idx == 0:
                    ax.set_title(f"Layer {col_idx}", fontsize=9)
                ax.set_xlabel("Frac tokens active", fontsize=7)
                ax.tick_params(labelsize=7)
        fig.suptitle("Per-Neuron Utilization Distribution (mlp_hidden)", fontsize=12, y=1.01)
        plt.tight_layout()
        plt.show()

## 5. Cosine Similarity Heatmaps

Does sparsity also reduce neuron-neuron correlation in mlp_out?

Shows neuron-neuron cosine similarity for each run × layer, plus mean off-diagonal per layer.

In [ ]:
heatmap_runs = [r for r in AVAILABLE_RUNS if has_artifact(r, "cosine_sim_heatmaps.npz")]

if not heatmap_runs:
    print("No cosine_sim_heatmaps.npz found yet.")
else:
    sim_data = {}
    mean_off_diag_data = {}

    for run_name in heatmap_runs:
        npz = load_npz(run_name, "cosine_sim_heatmaps.npz")
        sim_data[run_name] = {int(k.split("_")[1]): npz[k] for k in npz.files}
        mods = {}
        for layer_idx, mat in sim_data[run_name].items():
            mask = ~np.eye(mat.shape[0], dtype=bool)
            mods[layer_idx] = float(np.abs(mat[mask]).mean())
        mean_off_diag_data[run_name] = mods

    # Grid
    n_runs = len(heatmap_runs)
    fig, axes = plt.subplots(n_runs, N_LAYERS, figsize=(N_LAYERS * 2.5, n_runs * 2.5), squeeze=False)
    for row_idx, run_name in enumerate(heatmap_runs):
        for col_idx in range(N_LAYERS):
            ax = axes[row_idx][col_idx]
            if col_idx not in sim_data[run_name]:
                ax.axis("off")
                continue
            mat = sim_data[run_name][col_idx]
            step = max(1, mat.shape[0] // 64)
            im = ax.imshow(mat[::step, ::step], vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
            if col_idx == 0:
                ax.set_ylabel(RUN_LABELS[run_name], fontsize=8)
            if row_idx == 0:
                ax.set_title(f"Layer {col_idx}", fontsize=9)
            ax.set_xticks([])
            ax.set_yticks([])
    fig.suptitle("Neuron-Neuron Cosine Similarity (mlp_out)", fontsize=12, y=1.01)
    plt.colorbar(im, ax=axes, shrink=0.4, label="Cosine similarity")
    plt.tight_layout()
    plt.show()

    # Mean off-diagonal line chart
    fig, ax = plt.subplots(figsize=(8, 4))
    for run_name in heatmap_runs:
        vals = [mean_off_diag_data[run_name].get(l, 0) for l in range(N_LAYERS)]
        ax.plot(range(N_LAYERS), vals, marker="o",
                label=RUN_LABELS[run_name], color=RUN_COLORS[run_name],
                linestyle=RUN_LINESTYLES[run_name])
    ax.set_xlabel("Layer")
    ax.set_ylabel("Mean |off-diagonal| cosine sim")
    ax.set_title("Neuron Correlation per Layer")
    ax.set_xticks(range(N_LAYERS))
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6. Top-k Activating Contexts per Neuron

Do sparse neurons become more monosemantic — consistently activating on a narrower set of contexts?

In [ ]:
topk_runs = [r for r in AVAILABLE_RUNS if has_artifact(r, "topk_contexts.json")]
topk_data = {r: load_json(r, "topk_contexts.json") for r in topk_runs}
print(f"Loaded top-k contexts for {len(topk_runs)} runs")

In [ ]:
def show_neuron_contexts(run_name, layer, neuron, top_n=5):
    label = RUN_LABELS.get(run_name, run_name)
    entries = topk_data[run_name].get(str(layer), {}).get(str(neuron), [])
    if not entries:
        print(f"No data for {label} layer={layer} neuron={neuron}")
        return
    print(f"\n{'='*60}")
    print(f"Run: {label}  |  Layer: {layer}  |  Neuron: {neuron}")
    print(f"{'='*60}")
    for entry in entries[:top_n]:
        print(f"  Rank {entry['rank']}  activation={entry['activation']:+.4f}")
        print(f"  Context: {entry['context_text'][:200]!r}")
        print()


# Example: compare neuron 0, layer 3 across runs
for run_name in topk_runs:
    show_neuron_contexts(run_name, layer=3, neuron=0, top_n=3)

In [ ]:
# Activation distribution for a given layer/neuron across runs
LAYER = 3
NEURON = 0

n = len(topk_runs)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), sharey=True)
if n == 1:
    axes = [axes]

for ax, run_name in zip(axes, topk_runs):
    entries = topk_data[run_name].get(str(LAYER), {}).get(str(NEURON), [])
    acts = [e["activation"] for e in entries]
    if acts:
        colors = [RUN_COLORS[run_name] if a >= 0 else "#cccccc" for a in acts]
        ax.barh(range(len(acts)), acts, color=colors)
        ax.set_yticks(range(len(acts)))
        ax.set_yticklabels([f"Rank {e['rank']}" for e in entries], fontsize=8)
    ax.set_xlabel("Activation")
    ax.set_title(f"{RUN_LABELS[run_name]}\nL{LAYER} N{NEURON}", fontsize=8)
    ax.axvline(0, color="k", linewidth=0.5)

plt.suptitle("Top-k Activation Values", fontsize=12)
plt.tight_layout()
plt.show()

## 7. Linear Probe Results (POS Tags)

Does sparsity help or hurt the quality of learned representations? A linear probe for POS tags measures how much syntactic information is linearly decodable from mlp_out.

In [ ]:
probe_runs = [r for r in AVAILABLE_RUNS if has_artifact(r, "probe_results.json")]

if not probe_runs:
    print("No probe_results.json found yet.")
else:
    probe_data = {r: load_json(r, "probe_results.json") for r in probe_runs}

    rows = []
    for run_name in probe_runs:
        d = probe_data[run_name]
        for layer_idx in range(N_LAYERS):
            key = str(layer_idx)
            if key in d:
                rows.append({
                    "Run": RUN_LABELS[run_name],
                    "Layer": layer_idx,
                    "Accuracy": d[key]["accuracy"],
                    "Macro F1": d[key]["macro_f1"],
                })
    display(pd.DataFrame(rows).to_string(index=False))

In [ ]:
if probe_runs:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    for run_name in probe_runs:
        d = probe_data[run_name]
        layers, accs, f1s = [], [], []
        for layer_idx in range(N_LAYERS):
            key = str(layer_idx)
            if key in d:
                layers.append(layer_idx)
                accs.append(d[key]["accuracy"])
                f1s.append(d[key]["macro_f1"])
        kw = dict(marker="o", label=RUN_LABELS[run_name],
                  color=RUN_COLORS[run_name], linestyle=RUN_LINESTYLES[run_name])
        axes[0].plot(layers, accs, **kw)
        axes[1].plot(layers, f1s, **kw)

    for ax, title, ylabel in zip(
        axes,
        ["Probe Accuracy by Layer", "Probe Macro F1 by Layer"],
        ["Accuracy", "Macro F1"],
    ):
        ax.set_xlabel("Layer")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.set_xticks(range(N_LAYERS))
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
if probe_runs:
    n = len(probe_runs)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 6), squeeze=False)

    for col_idx, run_name in enumerate(probe_runs):
        ax = axes[0][col_idx]
        d = probe_data[run_name]
        label_names = d.get("label_names", [])
        mat = np.zeros((N_LAYERS, len(label_names)))
        for layer_idx in range(N_LAYERS):
            key = str(layer_idx)
            if key in d:
                per_class = d[key].get("per_class_f1", {})
                for j, tag in enumerate(label_names):
                    mat[layer_idx, j] = per_class.get(tag, 0.0)
        sns.heatmap(
            mat, ax=ax,
            xticklabels=label_names,
            yticklabels=[f"L{i}" for i in range(N_LAYERS)],
            vmin=0, vmax=1, cmap="YlOrRd",
            annot=True, fmt=".2f", linewidths=0.5,
        )
        ax.set_title(f"Per-class F1 — {RUN_LABELS[run_name]}", fontsize=9)
        ax.set_xlabel("POS Tag")
        ax.set_ylabel("Layer")

    plt.tight_layout()
    plt.show()